# Value iteration

_Artificial Intelligence (CS221)_

**Guess the values, improve them with the best action, repeat until they settle.**

We want the best value for each state, not the value of some given policy.

     Value iteration starts with rough guesses and improves them round by round.

---

This notebook builds value iteration from scratch, one piece at a time. We'll set up a tiny Markov decision process (MDP), sweep its values until they stop changing, and read off the optimal policy — then watch the same algorithm decide what every cell of a warehouse floor is worth and where a robot should go. Run each cell, read the note above it, and experiment with the rewards and discount. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Define the MDP

A Markov decision process is described by its **transitions** and **rewards**. Here we have 3 states and 2 actions. `T[a, s, s']` is the probability of landing in state `s'` when you take action `a` in state `s`; each action is *noisy* (action 0 mostly stays/advances one way, action 1 the other). State 2 is an **absorbing goal** worth a reward of 10 — once there, you stay there. The discount `gamma = 0.9` makes rewards that arrive sooner worth more than the same reward later.

In [ ]:
import numpy as np

# 3 states, 2 actions. T[a, s, s']; state 2 is the absorbing goal (reward 10).
T = np.array([
    [[0.8, 0.2, 0.0], [0.0, 0.8, 0.2], [0.0, 0.0, 1.0]],   # action 0
    [[0.2, 0.8, 0.0], [0.0, 0.2, 0.8], [0.0, 0.0, 1.0]],   # action 1
])

R = np.array([0.0, 0.0, 10.0])   # reward for being in each state
gamma = 0.9                      # discount factor

V = np.zeros(3)                  # initial value guess: zero for every state

### Step 2 — Sweep the values until they settle

Value iteration repeatedly applies the **Bellman optimality update**. For each action we compute `Q[a, s]`, the expected value of taking action `a` in state `s`: the discounted value of wherever that action might send us. Taking the **best** action per state (`Q.max(axis=0)`) gives the improved value `newV`. We repeat this sweep; once the largest change across all states falls below a tiny threshold, the values have **converged** to the optimal values `V*` and we stop.

In [ ]:
for it in range(40):
    Q = T @ (R + gamma * V)             # Q[a, s] = expected value of action a in state s
    newV = Q.max(axis=0)                # best action's value, per state

    largest_change = np.max(np.abs(newV - V))
    if largest_change < 1e-6:
        print("converged after", it, "iterations")
        break

    V = newV

### Step 3 — Read off the optimal policy

The optimal **values** tell us what each state is worth; the optimal **policy** tells us what to *do*. We recompute the per-action values one last time and, for each state, pick the action with the highest value (`argmax`). That action is the best move from that state.

In [ ]:
Q = T @ (R + gamma * V)          # per-action values under the converged V
policy = Q.argmax(axis=0)        # best action in each state

print("optimal values V*:", np.round(V, 3))
print("optimal policy   :", policy)

## Visualize the data & results

_On a real 3x4 warehouse floor with a goal shelf and a hazard spill, what is each cell worth and where should the robot go?_

### Step 1 — Lay out the warehouse grid world

Now we scale the same idea up to a classic 3x4 grid world. One cell is a **wall** the robot can't enter, one is the **goal** shelf (reward +1), and one is a **hazard** spill (reward -1). Every other step costs a small `-0.04`, which gently pushes the robot to reach the goal quickly rather than wander. We also list the four movement actions as row/column offsets and a helper `ok` that rejects moves off the grid or into the wall.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ROWS, COLS = 3, 4
WALL, GOAL, HAZARD = (1, 1), (0, 3), (1, 3)
gamma, step_cost = 0.9, -0.04

# The four movement actions as (row, col) offsets.
acts = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}

def ok(s):
    # A cell is valid if it's on the grid and not the wall.
    return 0 <= s[0] < ROWS and 0 <= s[1] < COLS and s != WALL

states = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) != WALL]

def reward(s):
    if s == GOAL:
        return 1.0
    if s == HAZARD:
        return -1.0
    return step_cost

### Step 2 — Define the noisy transitions

Movement is **stochastic**: the robot goes where it intends only 80% of the time, and slips to one of the two perpendicular directions 10% each. `trans(s, a)` returns the distribution over next states for taking action `a` in state `s`. Any move that would hit a wall or leave the grid leaves the robot where it was, so that probability mass is added back to the current cell.

In [ ]:
def trans(s, a):
    # 0.8 chance of the intended move, 0.1 each to the two perpendicular directions.
    if a in ("up", "down"):
        perp = ["left", "right"]
    else:
        perp = ["up", "down"]

    outcomes = [(0.8, a), (0.1, perp[0]), (0.1, perp[1])]

    res = {}
    for p, mv in outcomes:
        d = acts[mv]
        ns = (s[0] + d[0], s[1] + d[1])
        if not ok(ns):
            ns = s                    # blocked move -> stay in place
        res[ns] = res.get(ns, 0) + p
    return res

### Step 3 — Run value iteration on the grid

Same Bellman update as the toy MDP, just written with dictionaries. We start every cell at value 0, then sweep: the goal and hazard cells are terminal so they just hold their reward, while every other cell takes its own reward plus the discounted value of the **best** action (the action whose noisy outcomes have the highest expected next value). We keep sweeping until the largest change `delta` is negligible.

In [ ]:
V = {s: 0.0 for s in states}

for sweep in range(1000):
    nV = {}
    delta = 0
    for s in states:
        if s in (GOAL, HAZARD):
            nV[s] = reward(s)         # terminal cells just hold their reward
            continue

        action_values = []
        for a in acts:
            expected_next = sum(p * V[ns] for ns, p in trans(s, a).items())
            action_values.append(expected_next)

        nV[s] = reward(s) + gamma * max(action_values)
        delta = max(delta, abs(nV[s] - V[s]))

    V = nV
    if delta < 1e-8:
        break

### Step 4 — Visualize the value of each cell

Finally we drop the converged values into a grid and draw it as a heatmap, printing each cell's value on top (the wall cell stays blank). Brighter cells are worth more: you'll see values rise as they approach the goal shelf and dip near the hazard spill — exactly the landscape the robot should climb toward the goal while steering clear of the spill.

In [ ]:
grid = np.full((ROWS, COLS), np.nan)
for s in states:
    grid[s] = V[s]

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(grid, cmap="viridis")

for s in states:
    ax.text(s[1], s[0], round(V[s], 2), ha="center", va="center", color="white")

ax.set_title("warehouse floor: optimal value V* per cell")
fig.colorbar(im, ax=ax)
plt.show()